In [102]:
from dotenv import load_dotenv

In [103]:
load_dotenv()


True

In [104]:
from langchain_groq import ChatGroq

In [105]:
llm = ChatGroq(model="deepseek-r1-distill-llama-70b")

In [106]:
print(llm.invoke("What is the capital of France?").content)

<think>

</think>

The capital of France is Paris.


In [107]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [108]:
embedding_model=GoogleGenerativeAIEmbeddings(
    model="models/embedding-001"
)

In [109]:
doc_vector=embedding_model.embed_query("What is the capital of France?")  # Example usage

### 1. Data Ingestion

In [110]:
from langchain.document_loaders import PyPDFLoader

In [111]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [112]:
import os

In [113]:
file_path=os.path.join(os.getcwd(), "data", "sample.pdf")

In [114]:
loader=PyPDFLoader(file_path)

In [26]:
documents=loader.load()

### this is experimental. There is no deterministic way to split the text.

In [115]:

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150,
    length_function=len
)

In [116]:
docs = text_splitter.split_documents(documents)

In [117]:
len(docs)

765

In [118]:
docs[0].page_content

'Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗ Louis Martin† Kevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev'

In [119]:
## docs[0].metadata

In [120]:
from langchain.vectorstores import FAISS

In [121]:
len(embedding_model.embed_documents(docs[0].page_content)[0])

768

In [122]:
vectorstore=FAISS.from_documents(
    docs, embedding_model)

## This is a Retrieval process

means from the vector database  we are going to fetch or retrieve or rank the most appropriate k result

In [123]:
relevant_doc=vectorstore.similarity_search("llama2 finetuning benchmark experiments.")

In [124]:
relevant_doc=vectorstore.similarity_search("llama2 finetuning benchmark experiments.", k=10)

In [125]:
## relevant_doc

In [126]:
## relevant_doc[1].page_content

In [127]:
retriever=vectorstore.as_retriever(search_kwargs={"k": 10})

In [141]:
results=retriever.invoke("llama2 finetuning benchmark experiments.")

## Context: based on the question, retrieving the info from the vector database
## Question: This is a user question


In [129]:
prompt_template = """Answer the question based on the context provided below. 
If the context does not contain sufficient information, respond with:
"I do not have enough information about this."

Context: {context}
Question: {question}
Answer:"""

In [130]:
from langchain.prompts import PromptTemplate

In [131]:
prompt=PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [132]:
#prompt

In [134]:
from langchain_core.output_parsers import StrOutputParser

In [135]:
parser=StrOutputParser()

In [136]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [137]:
from langchain_core.runnables import RunnablePassthrough

In [138]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [140]:
output=rag_chain.invoke("tell me about llama2 finetuning benchmark experiments?")